[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/Zgraph/blob/main/zgraph/examples/Extruded_dimension.ipynb)

In [1]:
# Install Zgraph if running in Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/themintlab/Zgraph.git
    print("Successfully installed Zgraph!")

In [2]:
import torch
from zgraph.core import FactorNode, SignalNode, SignalNodes, ProductNode, ConstantNode, BaseLeafNode, DynamicLeafNode
from zgraph.transforms import finalize, legendre_transform
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
T, mu1, mu2, pa = SignalNodes(0, 1, 2, 3)

In [4]:
R = 8.314
RT = FactorNode([[R]], [T])
mu1A = FactorNode([2, -1], [RT, mu1] )
mu2A = FactorNode([-1], [mu2])
mu1B = FactorNode([-1], [mu1])
mu2B = FactorNode([1, -1], [RT, mu2] )

In [5]:
phaseA = FactorNode(torch.eye(2), [mu1A, mu2A], beta=RT)
phaseB = FactorNode(torch.eye(2), [mu1B, mu2B], beta=RT)

In [6]:
pb = FactorNode([1, -1], [ConstantNode(1), pa])
system = FactorNode([1,1], [ProductNode([pa, phaseA]), ProductNode([pb, phaseB])], beta=0)

In [7]:
phaseAb, phaseBb, systemb = finalize([phaseA, phaseB, system], compile_graph=False)

fcns = legendre_transform([phaseA, phaseB, system], [1, 2])
fA, fB, fsys = finalize(fcns, compile_graph=False)

In [8]:
T_val = torch.tensor(298.15)
mu1_range = torch.linspace(-10*R*300, 10*R*300, steps=100)
pa_range = torch.linspace(0., 1., steps=10)
mu2_range = -mu1_range

MU1_grid, PA_grid = torch.meshgrid(mu1_range, pa_range, indexing="ij")

MU1_flat = MU1_grid.flatten()
PA_flat = PA_grid.flatten()

T_flat = T_val.expand_as(MU1_flat)
MU2_flat = -MU1_flat

input_tensor = torch.stack([T_flat, MU1_flat, MU2_flat, PA_flat], dim=-1)

In [9]:
fAd, cAd = fA(input_tensor)
fBd, cBd = fB(input_tensor)
fsysd, csysd = fsys(input_tensor)

# Grand potential plot

In [10]:
# Rebuild a dedicated 2D input tensor over (mu1, pa) for surface plots
T_surface = T_val.expand_as(MU1_flat)
MU2_surface = -MU1_flat
input_tensor_surface = torch.stack([T_surface, MU1_flat, MU2_surface, PA_flat], dim=-1)

mu1_axis = MU1_grid[:, 0].detach().cpu().numpy()
pa_axis = PA_grid[0, :].detach().cpu().numpy()

In [11]:
# Evaluate grand potentials on the same (mu1, pa) grid
gA_grid = phaseAb(input_tensor_surface).detach().cpu().numpy().reshape(MU1_grid.shape)
gB_grid = phaseBb(input_tensor_surface).detach().cpu().numpy().reshape(MU1_grid.shape)
gsys_grid = systemb(input_tensor_surface).detach().cpu().numpy().reshape(MU1_grid.shape)

# Free energies and transformed coordinates from the Legendre transform
fA_grid = (-fAd.detach().cpu().numpy()).reshape(MU1_grid.shape)
fB_grid = (-fBd.detach().cpu().numpy()).reshape(MU1_grid.shape)
fsys_grid = (-fsysd.detach().cpu().numpy()).reshape(MU1_grid.shape)

xA_tr_grid = (-cAd[:, 1].detach().cpu().numpy()).reshape(MU1_grid.shape)
xB_tr_grid = (-cBd[:, 1].detach().cpu().numpy()).reshape(MU1_grid.shape)
xsys_tr_grid = (-csysd[:, 1].detach().cpu().numpy()).reshape(MU1_grid.shape)

pa_grid_np = PA_grid.detach().cpu().numpy()

# Free energy plot

In [12]:
fig = make_subplots(
    rows=2, cols=1,
    specs=[[{"type": "surface"}], [{"type": "surface"}]],
    vertical_spacing=0.06,
    subplot_titles=("Grand potential", "Free energy (Legendre coordinates)")
)

colors = {
    "phase A": "#1f77b4",
    "phase B": "#ff7f0e",
    "Equilibrium": "#2ca02c",
}

mu1_line = mu1_axis

# pa=1 boundary for phase A (last pa index)
gA_line = gA_grid[:, -1]
fA_line = fA_grid[:, -1]
xA_line = xA_tr_grid[:, -1]
pa1_line = [float(pa_axis[-1])] * len(mu1_line)

# pa=0 boundary for phase B (first pa index)
gB_line = gB_grid[:, 0]
fB_line = fB_grid[:, 0]
xB_line = xB_tr_grid[:, 0]
pa0_line = [float(pa_axis[0])] * len(mu1_line)

# System grand potential surface in (mu1, pa, g)
fig.add_trace(
    go.Surface(
        x=mu1_axis,
        y=pa_axis,
        z=gsys_grid.T,
        name="Equilibrium",
        legendgroup="Equilibrium",
        showscale=False,
        colorscale="Viridis",
        opacity=0.85,
    ),
    row=1, col=1,
 )

fig.add_trace(
    go.Scatter3d(
        x=mu1_line,
        y=pa1_line,
        z=gA_line,
        mode="lines",
        name="phase A",
        legendgroup="phase A",
        line=dict(width=8, color=colors["phase A"]),
    ),
    row=1, col=1,
 )

fig.add_trace(
    go.Scatter3d(
        x=mu1_line,
        y=pa0_line,
        z=gB_line,
        mode="lines",
        name="phase B",
        legendgroup="phase B",
        line=dict(width=8, color=colors["phase B"]),
    ),
    row=1, col=1,
 )

# Free-energy panel in transformed Legendre coordinates (x_tr, pa, f)
fig.add_trace(
    go.Surface(
        x=xsys_tr_grid.T,
        y=pa_grid_np.T,
        z=fsys_grid.T,
        name="Equilibrium",
        legendgroup="Equilibrium",
        showlegend=False,
        showscale=False,
        colorscale="Viridis",
        opacity=0.85,
    ),
    row=2, col=1,
 )

fig.add_trace(
    go.Scatter3d(
        x=xA_line,
        y=pa1_line,
        z=fA_line,
        mode="lines",
        name="phase A",
        legendgroup="phase A",
        showlegend=False,
        line=dict(width=8, color=colors["phase A"]),
    ),
    row=2, col=1,
 )

fig.add_trace(
    go.Scatter3d(
        x=xB_line,
        y=pa0_line,
        z=fB_line,
        mode="lines",
        name="phase B",
        legendgroup="phase B",
        showlegend=False,
        line=dict(width=8, color=colors["phase B"]),
    ),
    row=2, col=1,
 )

camera = dict(eye=dict(x=1.6, y=1.6, z=0.8))

fig.update_scenes(
    xaxis_title="Chemical potential, μ1",
    yaxis_title="Phase fraction, pa",
    zaxis_title="Grand potential, Ω",
    row=1, col=1,
    camera=camera,
 )

fig.update_scenes(
    xaxis_title="Transformed coordinate, -∂Ω/∂μ",
    yaxis_title="Phase fraction, pa",
    zaxis_title="Free energy",
    row=2, col=1,
    camera=camera,
 )

fig.update_layout(
    height=1100,
    width=900,
    template="plotly_white",
    legend_title_text="Phase",
    # Keep both 3D scenes on the same interactive state/camera baseline
    uirevision="sync-3d",
    scene=dict(dragmode="turntable"),
    scene2=dict(
        dragmode="turntable",
        camera=dict(
            eye=dict(x=1.6, y=1.6, z=0.8),
        ),
    ),
)

fig.show()